In [1]:
import pandas as pd
import numpy as np

In [2]:
#define a population for S and I people with high and low activities across three counties
pop_size = pd.DataFrame({
    'InfState': ['S', 'S', 'I', 'I', 
                 'S', 'S', 'I', 'I', 
                 'S', 'S', 'I', 'I'],
    'County': pd.Categorical(['Orange_HA', 'Orange_LA', 'Orange_HA', 'Orange_LA', 
                              'Durham_HA', 'Durham_LA', 'Durham_HA', 'Durham_LA', 
                              'Chatham_HA', 'Chatham_LA', 'Chatham_HA', 'Chatham_LA'], 
                             categories=['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']),
    'N': [38, 345, 1, 0, 
          92, 831, 0, 0, 
          18, 165, 0, 0],
    'T': 2025
})
pop_size

,InfState,County,N,T
0,S,Orange_HA,38,2025
1,S,Orange_LA,345,2025
2,I,Orange_HA,1,2025
3,I,Orange_LA,0,2025
4,S,Durham_HA,92,2025
5,S,Durham_LA,831,2025
6,I,Durham_HA,0,2025
7,I,Durham_LA,0,2025
8,S,Chatham_HA,18,2025
9,S,Chatham_LA,165,2025


#### Contact Matrix and Geo Matrix

In [3]:
#contact rate matrix, Beta: High Activity, Low Activity
contact_rate_matrix = np.array([[0.1428, 0.0049], [0.00055, 0.0077]])
contact_rate_matrix

array([[0.1428 , 0.0049 ],
       [0.00055, 0.0077 ]])

In [4]:
#Geographcial mixing matrix, T: Orange, Durham, Chatham
geo_mixing_matrix = np.array([[0.7, 0.1869, 0.1131], [0.1851, 0.7, 0.1150], [0.2210, 0.07895, 0.7]])
geo_mixing_matrix

array([[0.7    , 0.1869 , 0.1131 ],
       [0.1851 , 0.7    , 0.115  ],
       [0.221  , 0.07895, 0.7    ]])

#### Kronecker Matrix

In [5]:
#transmission rate of high and low activities by geo
transmission_by_geo = np.kron(geo_mixing_matrix, contact_rate_matrix)
transmission_by_geo = np.round(transmission_by_geo, decimals=5)
transmission_by_geo

array([[9.996e-02, 3.430e-03, 2.669e-02, 9.200e-04, 1.615e-02, 5.500e-04],
       [3.800e-04, 5.390e-03, 1.000e-04, 1.440e-03, 6.000e-05, 8.700e-04],
       [2.643e-02, 9.100e-04, 9.996e-02, 3.430e-03, 1.642e-02, 5.600e-04],
       [1.000e-04, 1.430e-03, 3.800e-04, 5.390e-03, 6.000e-05, 8.900e-04],
       [3.156e-02, 1.080e-03, 1.127e-02, 3.900e-04, 9.996e-02, 3.430e-03],
       [1.200e-04, 1.700e-03, 4.000e-05, 6.100e-04, 3.800e-04, 5.390e-03]])

In [6]:
#Transmission by Geo matrix
index_names= ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
column_names = ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
pd.DataFrame(transmission_by_geo, index=index_names, columns=column_names)

,Orange_HA,Orange_LA,Durham_HA,Durham_LA,Chatham_HA,Chatham_LA
Orange_HA,0.09996,0.00343,0.02669,0.00092,0.01615,0.00055
Orange_LA,0.00038,0.00539,0.00010,0.00144,0.00006,0.00087
Durham_HA,0.02643,0.00091,0.09996,0.00343,0.01642,0.00056
Durham_LA,0.00010,0.00143,0.00038,0.00539,0.00006,0.00089
Chatham_HA,0.03156,0.00108,0.01127,0.00039,0.09996,0.00343
Chatham_LA,0.00012,0.00170,0.00004,0.00061,0.00038,0.00539


#### Infection Array of All Groups

In [7]:
#infection array
inf_array = pop_size.loc[pop_size['InfState']=='I'].groupby(['County'], observed=False)['N'].sum(numeric_only=True).values
inf_array

array([1, 0, 0, 0, 0, 0])

#### Vaccination Effect integrated to Transmission Rate

In [8]:
#Transmission by Vaccination Effect and Geo matrix
#(1 - Ve_1) * Orange_HA(#1, #2, #3...#6)
#(1 - Ve_2) * Orange_LA(#1, #2, #3...#6)
VaccEffect_input = np.array([0.7, 0.0, 0.7, 0.0, 0.7, 0.0])
Vacc_Trans_Geo = (1- VaccEffect_input)[:, np.newaxis] * transmission_by_geo
index_names= ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
column_names = ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
pd.DataFrame(Vacc_Trans_Geo, index=index_names, columns=column_names)

,Orange_HA,Orange_LA,Durham_HA,Durham_LA,Chatham_HA,Chatham_LA
Orange_HA,0.029988,0.001029,0.008007,0.000276,0.004845,0.000165
Orange_LA,0.000380,0.005390,0.000100,0.001440,0.000060,0.000870
Durham_HA,0.007929,0.000273,0.029988,0.001029,0.004926,0.000168
Durham_LA,0.000100,0.001430,0.000380,0.005390,0.000060,0.000890
Chatham_HA,0.009468,0.000324,0.003381,0.000117,0.029988,0.001029
Chatham_LA,0.000120,0.001700,0.000040,0.000610,0.000380,0.005390


#### Probability of Survival

In [9]:
#Probability of Survival calculated by Vaccination Effect associated with Transmission by Geo
prI_survival = np.power(np.exp(-1 * Vacc_Trans_Geo), inf_array)
prI_survival

array([[0.97045718, 1.        , 1.        , 1.        , 1.        ,
        1.        ],
       [0.99962007, 1.        , 1.        , 1.        , 1.        ,
        1.        ],
       [0.99210235, 1.        , 1.        , 1.        , 1.        ,
        1.        ],
       [0.9999    , 1.        , 1.        , 1.        , 1.        ,
        1.        ],
       [0.99057668, 1.        , 1.        , 1.        , 1.        ,
        1.        ],
       [0.99988001, 1.        , 1.        , 1.        , 1.        ,
        1.        ]])

#### Probability of Infection

In [10]:
#Probability of Infection
prI = 1 - prI_survival.prod(axis=1) #using 1- prI is questionable, maybe we should just use prI, see prI_2 below for different math
prI

array([2.95428210e-02, 3.79927809e-04, 7.89764840e-03, 9.99950002e-05,
       9.42331961e-03, 1.19992800e-04])

#### Deltas of Infected People - stochastic and deterministic

In [11]:
#suspectible mask based on initial population
is_susceptible = (pop_size['InfState'] == 'S')
is_susceptible

0      True
1      True
2     False
3     False
4      True
5      True
6     False
7     False
8      True
9      True
10    False
11    False
Name: InfState, dtype: bool

In [12]:
#number of susceptible people with high and low activities in each county
deltas = pop_size[is_susceptible].copy()
deltas

,InfState,County,N,T
0,S,Orange_HA,38,2025
1,S,Orange_LA,345,2025
4,S,Durham_HA,92,2025
5,S,Durham_LA,831,2025
8,S,Chatham_HA,18,2025
9,S,Chatham_LA,165,2025


In [13]:
present_category_codes = pop_size['County'].cat.codes.to_numpy()
present_category_codes

array([0, 1, 0, 1, 2, 3, 2, 3, 4, 5, 4, 5], dtype=int8)

In [14]:
susceptible_group_codes = present_category_codes[is_susceptible.to_numpy()]
susceptible_group_codes

array([0, 1, 2, 3, 4, 5], dtype=int8)

In [15]:
#prI of each group, i.e. prI for high and low activities of each county
prI_per_group = prI[susceptible_group_codes]
prI_per_group

array([2.95428210e-02, 3.79927809e-04, 7.89764840e-03, 9.99950002e-05,
       9.42331961e-03, 1.19992800e-04])

In [ ]:
#stochastic
N_stochastic = -np.random.binomial(deltas["N"], prI_per_group)
deltas_stochastic = deltas.assign(N=N_stochastic)
deltas_stochastic

,InfState,County,N,T
0,S,Orange_HA,-1,2025
1,S,Orange_LA,0,2025
4,S,Durham_HA,-1,2025
5,S,Durham_LA,0,2025
8,S,Chatham_HA,0,2025
9,S,Chatham_LA,0,2025


In [17]:
#deterministic
#infected people with high and low activities in each county
N_deterministic = -deltas['N'] * prI_per_group
deltas_deterministic = deltas.assign(N=N_deterministic)
deltas_deterministic #deltas means the number of people moving from S to I state

,InfState,County,N,T
0,S,Orange_HA,-1.122627,2025
1,S,Orange_LA,-0.131075,2025
4,S,Durham_HA,-0.726584,2025
5,S,Durham_LA,-0.083096,2025
8,S,Chatham_HA,-0.169620,2025
9,S,Chatham_LA,-0.019799,2025


#### Population Size Updates after One-time Transmission

In [18]:
#deterministic
new_infected = deltas_deterministic.assign(InfState=deltas['InfState'].replace('S', 'I'), N=-deltas_deterministic['N'])
new_infected

,InfState,County,N,T
0,I,Orange_HA,1.122627,2025
1,I,Orange_LA,0.131075,2025
4,I,Durham_HA,0.726584,2025
5,I,Durham_LA,0.083096,2025
8,I,Chatham_HA,0.169620,2025
9,I,Chatham_LA,0.019799,2025


In [19]:
pop_size_updated = pd.concat([pop_size, deltas_deterministic, new_infected]).groupby(['InfState', 'County'], observed=True).agg({'N': 'sum', 'T': 'max'}).reset_index(drop=False)
pop_size_updated

,InfState,County,N,T
0,I,Orange_HA,2.122627,2025
1,I,Orange_LA,0.131075,2025
2,I,Durham_HA,0.726584,2025
3,I,Durham_LA,0.083096,2025
4,I,Chatham_HA,0.169620,2025
5,I,Chatham_LA,0.019799,2025
6,S,Orange_HA,36.877373,2025
7,S,Orange_LA,344.868925,2025
8,S,Durham_HA,91.273416,2025
9,S,Durham_LA,830.916904,2025


#### Recovered Population

In [20]:
#Adding compartment R
current_infected = pop_size_updated[pop_size_updated['InfState']=='I']
current_infected

,InfState,County,N,T
0,I,Orange_HA,2.122627,2025
1,I,Orange_LA,0.131075,2025
2,I,Durham_HA,0.726584,2025
3,I,Durham_LA,0.083096,2025
4,I,Chatham_HA,0.169620,2025
5,I,Chatham_LA,0.019799,2025


In [21]:
recover_rate = 1/14
recovered = current_infected.assign(InfState=current_infected['InfState'].replace('I', 'R'), N=current_infected["N"] * (1 - np.exp(-1*recover_rate)))
recovered

,InfState,County,N,T
0,R,Orange_HA,0.146328,2025
1,R,Orange_LA,0.009036,2025
2,R,Durham_HA,0.050089,2025
3,R,Durham_LA,0.005728,2025
4,R,Chatham_HA,0.011693,2025
5,R,Chatham_LA,0.001365,2025
